# ChiralFold — Expanded Ramachandran / MolProbity Benchmark (n ≥ 100)

This notebook reproduces the **expanded** ChiralFold vs wwPDB/MolProbity Ramachandran
benchmark on a stratified random sample of ≥100 PDB structures (X-ray ultra/high/med/low
resolution + NMR + cryo-EM).

It will:
1. Clone the ChiralFold repository.
2. Install the package with its `dev` extras.
3. Run the benchmark **dry-run** (plan only, no downloads) to verify the sampler.
4. Run the **full** benchmark with a Google Drive–backed cache so downloads survive
   runtime restarts (or a local cache if Drive is not mounted).
5. Inspect the summary JSON.
6. Zip the outputs and download them.

**Notes**
- The full run downloads ~110 PDB / mmCIF files plus wwPDB validation reports and audits
  each. On a fresh machine with no cache this can take a while depending on network speed.
- A successful run writes `ramachandran_100struct_summary.json` with `n_success`; the run
  is considered clean when `n_success >= 100`.
- This notebook writes outputs under an `--out-prefix` and therefore **does not overwrite**
  the canonical `results/molprobity_comparison.json` (the historical 31-structure result).

## 1. (Optional) Mount Google Drive for a persistent download cache

Mounting Drive lets the PDB / validation-XML cache survive runtime disconnects, so you can
`--resume` without re-downloading. If you skip this (or run outside Colab), a local cache
directory is used instead.

In [ ]:
import os

CACHE_DIR = None
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/chiralfold_ramachandran_cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    print('Using Google Drive cache:', CACHE_DIR)
except Exception as exc:  # noqa: BLE001
    CACHE_DIR = os.path.abspath('ramachandran_100struct_cache')
    os.makedirs(CACHE_DIR, exist_ok=True)
    print('Drive not mounted (', exc, '); using local cache:', CACHE_DIR)

## 2. Clone the repository and install ChiralFold with dev extras

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/Tommaso-R-Marena/ChiralFold.git'
REPO_DIR = '/content/ChiralFold' if os.path.isdir('/content') else os.path.abspath('ChiralFold')

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
import chiralfold
print('ChiralFold version:', chiralfold.__version__)

## 3. Dry-run: verify the stratified sampler resolves all bins

In [ ]:
import subprocess, sys

OUT_PREFIX = 'results/ramachandran_100struct'
SEED = 20260514
N = 120  # oversample so >=100 usable structures survive attrition

subprocess.run([
    sys.executable, 'benchmarks/expand_ramachandran_benchmark.py',
    '--dry-run', '--n', str(N), '--seed', str(SEED),
    '--cache-dir', CACHE_DIR, '--out-prefix', OUT_PREFIX, '--resume',
], check=True)

## 4. Full run (network + compute)

Downloads + audits every structure in the plan, writing `*_comparison.csv`,
`*_summary.json`, and `*_plot.png` under the output prefix. `--resume` reuses anything
already present in the cache. Output is streamed to a log file as well.

In [ ]:
import subprocess, sys

LOG = 'results/ramachandran_100struct_run.log'
os.makedirs('results', exist_ok=True)
with open(LOG, 'w') as logfp:
    proc = subprocess.run([
        sys.executable, '-u', 'benchmarks/expand_ramachandran_benchmark.py',
        '--n', str(N), '--seed', str(SEED),
        '--cache-dir', CACHE_DIR, '--out-prefix', OUT_PREFIX, '--resume',
    ], stdout=logfp, stderr=subprocess.STDOUT)
print('Exit code:', proc.returncode)
print('--- last 25 log lines ---')
print('\n'.join(open(LOG).read().splitlines()[-25:]))

## 5. Inspect the summary JSON

A clean run has `n_success >= 100` with Spearman ρ and p-value present. Any skipped
structures are documented under `failures`.

In [ ]:
import json

summary = json.load(open(f'{OUT_PREFIX}_summary.json'))
print('n_success :', summary['n_success'])
print('n_planned :', summary['n_planned'])
print('n_failures:', summary['n_failures'])
print('Spearman rho:', round(summary['spearman_rho'], 3),
      'p =', summary['spearman_p_value'])
print('Pearson r   :', round(summary['pearson_r'], 3),
      'p =', summary['pearson_p_value'])
print('ChiralFold mean outlier %:', round(summary['chiralfold_mean_outlier_pct'], 2))
print('wwPDB mean outlier %     :', round(summary['wwpdb_mean_outlier_pct'], 2))
if summary['n_success'] >= 100:
    print('\nCLEAN RUN: n_success >= 100')
else:
    print('\nINCOMPLETE: n_success < 100 — do NOT update canonical numbers.')

from IPython.display import Image, display
display(Image(f'{OUT_PREFIX}_plot.png'))

## 6. Zip the outputs and download

In [ ]:
import zipfile, os

ZIP_PATH = 'ramachandran_100struct_outputs.zip'
outputs = [
    f'{OUT_PREFIX}_comparison.csv',
    f'{OUT_PREFIX}_summary.json',
    f'{OUT_PREFIX}_plot.png',
    'results/ramachandran_100struct_run.log',
]
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in outputs:
        if os.path.exists(f):
            zf.write(f, arcname=os.path.basename(f))
print('Wrote', ZIP_PATH)

try:
    from google.colab import files  # type: ignore
    files.download(ZIP_PATH)
except Exception as exc:  # noqa: BLE001
    print('Not in Colab; find the zip at', os.path.abspath(ZIP_PATH), '(', exc, ')')

## 7. (Optional) Larger ~200-structure representative run

This optional run uses **seeded era-representative sampling** (candidates drawn across
the full RCSB result range per bin, not just the oldest entries) and targets ~200+
usable structures. It is **slower** than the run above because most structures are
fresh downloads (including larger modern / cryo-EM entries), so it is presented as
optional rather than the default. It writes to a separate `ramachandran_200struct`
prefix and a separate cache, leaving the smaller-run and canonical results untouched.

The historical pilot result from this command is n_success = 279, Spearman ρ = 0.48
(95% CI [0.36, 0.58]), Pearson r = 0.80 — consistent with the smaller runs.

In [ ]:
import os, subprocess, sys

# Persistent cache on Drive if mounted, else local.
if os.path.isdir('/content/drive/MyDrive'):
    CACHE_200 = '/content/drive/MyDrive/chiralfold_ramachandran_200struct_cache'
else:
    CACHE_200 = os.path.abspath('ramachandran_200struct_cache')
os.makedirs(CACHE_200, exist_ok=True)
os.makedirs('results', exist_ok=True)

OUT_PREFIX_200 = 'results/ramachandran_200struct'
LOG_200 = 'results/ramachandran_200struct_run.log'

with open(LOG_200, 'w') as logfp:
    proc = subprocess.run([
        sys.executable, '-u', 'benchmarks/expand_ramachandran_benchmark.py',
        '--n', '220', '--seed', '20260612',
        '--cache-dir', CACHE_200, '--out-prefix', OUT_PREFIX_200, '--resume',
    ], stdout=logfp, stderr=subprocess.STDOUT)
print('Exit code:', proc.returncode)
print('\n'.join(open(LOG_200).read().splitlines()[-15:]))

import json
s = json.load(open(f'{OUT_PREFIX_200}_summary.json'))
print('\nn_success:', s['n_success'], '| Spearman rho:', round(s['spearman_rho'], 3),
      '| Pearson r:', round(s['pearson_r'], 3))
print('Spearman 95% CI:', s.get('spearman_rho_ci95'),
      '| Pearson 95% CI:', s.get('pearson_r_ci95'))
from IPython.display import Image, display
display(Image(f'{OUT_PREFIX_200}_plot.png'))